# PrivaDE workflow

In this scenario, we assume Bob has multiple data points to contribute to Alice's ML model. Alice is trying to value the dataset as a whole, judging on the diversity, uncertainty of the datasets as well as the current model's performance on the dataset. Moreover, the parties are assumed to be malicious, which means they might deviate from the protocol to maximize their own utility.

## Part 0: Setup

We set up Alice's model and Bob's dataset.

In [1]:
import os
import torch
import sys
import random
sys.path.append('..')  # Add privade directory to path
from privade.data import get_dataset
from privade.models import get_model
import numpy as np
from torch.utils.data import DataLoader, TensorDataset
import torch.nn as nn
import torch.optim as optim

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

N = 1000 #Bob's dataset size

full_model = get_model('lenet5', 'mnist')
full_data = get_dataset('mnist')

# Randomly select 1000 images as Bob's dataset
indices = random.sample(range(len(full_data)), N)
bob_images = np.array([full_data[i][0].numpy() for i in indices])
bob_labels = np.array([full_data[i][1]  for i in indices])

# Ramdomly select 1000 images as Alice's initial dataset
indices = random.sample(range(len(full_data)), N)
alice_images = np.array([full_data[i][0].numpy() for i in indices])
alice_labels = np.array([full_data[i][1]  for i in indices])

# Train the model for a few epochs
alice_images_tensor = torch.FloatTensor(alice_images)
alice_labels_tensor = torch.LongTensor(alice_labels)
alice_dataset = TensorDataset(alice_images_tensor, alice_labels_tensor)
alice_dataloader = DataLoader(alice_dataset, batch_size=32, shuffle=True)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(full_model.parameters(), lr=0.001)
full_model.train()
for epoch in range(5):
    running_loss = 0.0
    for images, labels in alice_dataloader:
        optimizer.zero_grad()
        outputs = full_model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
    print(f'Epoch {epoch+1}/5, Loss: {running_loss/len(alice_dataloader):.4f}')
    
# Save models and datasets
torch.save(bob_images, 'data/bob_images.pth')
torch.save(bob_labels, 'data/bob_labels.pth')
torch.save(full_model.state_dict(), 'data/alice_full_model.pth')

Epoch 1/5, Loss: 1.5698
Epoch 2/5, Loss: 0.6102
Epoch 3/5, Loss: 0.4255
Epoch 4/5, Loss: 0.3455
Epoch 5/5, Loss: 0.2885


### Model distillation

For Alice's preprocessing, she has an optional step to do model distillation, to obtain a smaller model for data evaluation.

In [2]:
from privade.distillation import train_distilled_model

student_model = get_model('lenetxs', 'mnist')

kd_train_loader = DataLoader(alice_dataset, batch_size=32, shuffle=True)

trained_student_model = train_distilled_model(full_model, student_model, kd_train_loader, kd_train_loader,epochs=10)

torch.save(trained_student_model.state_dict(), "data/alice_student_model.pth")

Starting knowledge distillation training on cuda
Epochs: 10, LR: 0.01, Alpha: 0.7, Temperature: 4.0
Epoch: [0][0/32] Loss 4.4842 (4.4842) Acc@1 6.250 (6.250)
Epoch: [0][10/32] Loss 4.2170 (4.1961) Acc@1 15.625 (14.773)


Epoch: [0][20/32] Loss 2.0530 (3.8149) Acc@1 71.875 (23.363)
Epoch: [0][30/32] Loss 1.8215 (3.3393) Acc@1 56.250 (33.468)
Epoch [1/10] - Train Loss: 3.3211 (CE: 1.8822, KD: 6.6784) Train Acc: 34.00%
Epoch: [1][0/32] Loss 1.6239 (1.6239) Acc@1 59.375 (59.375)
Epoch: [1][10/32] Loss 1.5442 (1.1503) Acc@1 65.625 (75.568)
Epoch: [1][20/32] Loss 0.8155 (1.0890) Acc@1 81.250 (77.976)
Epoch: [1][30/32] Loss 0.5903 (0.9583) Acc@1 87.500 (80.040)
Epoch [2/10] - Train Loss: 0.9592 (CE: 0.6142, KD: 1.7641) Train Acc: 80.00%
Epoch: [2][0/32] Loss 0.5229 (0.5229) Acc@1 87.500 (87.500)
Epoch: [2][10/32] Loss 0.3509 (0.4595) Acc@1 90.625 (92.614)
Epoch: [2][20/32] Loss 0.1896 (0.4203) Acc@1 100.000 (92.708)
Epoch: [2][30/32] Loss 0.4512 (0.3991) Acc@1 90.625 (92.641)
Epoch [3/10] - Train Loss: 0.4010 (CE: 0.2584, KD: 0.7338) Train Acc: 92.50%
Epoch: [3][0/32] Loss 0.5272 (0.5272) Acc@1 90.625 (90.625)
Epoch: [3][10/32] Loss 0.2769 (0.2674) Acc@1 93.750 (96.875)
Epoch: [3][20/32] Loss 0.1667 (0.2722) 

In [3]:
from torchsummary import summary
summary(trained_student_model, input_size=alice_images[0].shape)

----------------------------------------------------------------
        Layer (type)               Output Shape         Param #
            Conv2d-1            [-1, 3, 24, 24]              78
            Square-2            [-1, 3, 24, 24]               0
           Flatten-3                 [-1, 1728]               0
            Linear-4                  [-1, 900]       1,556,100
              ReLU-5                  [-1, 900]               0
            Linear-6                   [-1, 32]          28,832
              ReLU-7                   [-1, 32]               0
            Linear-8                   [-1, 10]             330
Total params: 1,585,340
Trainable params: 1,585,340
Non-trainable params: 0
----------------------------------------------------------------
Input size (MB): 0.00
Forward/backward pass size (MB): 0.05
Params size (MB): 6.05
Estimated Total Size (MB): 6.10
----------------------------------------------------------------


### Split Model

Alice also needs to perform split model to split her model $M$ into $A,B,C$. 


In [4]:
from privade.split import split_model
try:
    
    # Split the model
    model_A, model_B, model_C, split_stats = split_model(
        data_loader=alice_dataloader,
        model=trained_student_model,
    )
    model_A.to(device)
    model_B.to(device)
    model_C.to(device)

    print(f"\nSplit successful!")
    print(f"First activation layer: {split_stats['first_activation_layer']}")
    print(f"Optimal boundary layer: {split_stats['optimal_layer']}")
    print(f"Privacy preserved rate: {split_stats['privacy_preserved_rate']:.3f}")
    print(f"Client model (model_B): {len(list(model_B.children()))} layers")
    print(f"Server model (model_C): {len(list(model_C.children()))} layers")
    
except Exception as e:
    print(f"Error during splitting: {e}")
    import traceback
    traceback.print_exc()

Found first activation layer at index 1: Square
First activation layer:  1
Identifying candidate layers for B/C boundary...
Candidates [4, 0]
Found 1 candidate layers: [4]
Starting boundary analysis with DINA attack...
Attack epochs: 20
This may take a while...
Starting boundary layer evaluation...


Evaluating layers:   0%|          | 0/1 [00:00<?, ?it/s]


Evaluating layer 4...
    Split layer: 4
    Distillation taps (pre-ReLU conv), ordered near→far: [3]
    Sub-blocks: [[0, 1, 2, 3, 4]]
    Using linear inverse network
    Feature path: [900, 900]
    Final spatial size: (24, 24)
    Loss coefficients: [1.0, 3.0]
    Training DINA attack for 20 epochs...


DINA Epoch 1/20: 100%|██████████| 32/32 [00:00<00:00, 344.45it/s, Loss=19.7495]


Epoch 1: Average Loss = 23.1142


...

Epoch 19: Average Loss = 22.7936


DINA Epoch 20/20: 100%|██████████| 32/32 [00:00<00:00, 799.56it/s, Loss=18.5871]


Epoch 20: Average Loss = 22.6306
    Evaluating DINA attack...


Evaluating layers: 100%|██████████| 1/1 [00:01<00:00,  1.09s/it]

  Privacy Preserved Rate: 0.000

BOUNDARY ANALYSIS RESULTS

Layer 4:
  Privacy Preserve Rate: 0.000
  Attack Success: 1.000
  Avg SSIM: 0.511

OPTIMAL BOUNDARY LAYER: None
Using layer 4 as fallback.

Flattened model has 8 layers:
  0: Conv2d
  1: Square
  2: Flatten
  3: Linear
  4: ReLU
  5: Linear
  6: ReLU
  7: Linear

Three-Model Split Statistics:
Total layers: 8
Model A ends at layer: 1 (Square)
Model B: layers 2 to 4 (3 layers)
Model C: layers 5 to 7 (3 layers)
Privacy Rate: 0.000
Attack Success: 1.000
Average SSIM: 0.511

Model A layers (2):
  0: Conv2d
  1: Square

Model B layers (3):
  0: Flatten
  1: Linear
  2: ReLU

Model C layers (3):
  0: Linear
  1: ReLU
  2: Linear

Split successful!
First activation layer: 1
Optimal boundary layer: 4
Privacy preserved rate: 0.000
Client model (model_B): 3 layers
Server model (model_C): 3 layers


In [5]:
model_A.to(device)
model_A(torch.tensor(alice_images).to(device)).shape

torch.Size([1000, 3, 24, 24])

In [6]:
#Optionally, add a weight mixer to model A and unmix in model_B. For the experiment, we will not add it for now.
from privade.weight_mixer import weight_mixer

model_A, model_B = weight_mixer(model_A, model_B)

In [7]:
#Save all models

torch.save(model_A.state_dict(),"data/model_a.pth")
torch.save(model_B.state_dict(),"data/model_b.pth")
torch.save(model_C.state_dict(),"data/model_c.pth") 

## Part 1: Representative Set selection

Once all models are prepared, Bob can start selecting a representative set from his dataset.

In [8]:
#Optional step: Dimension reduction
from privade.dim_reduction import reduce_image_dimensions

target_dimension = 50
reduced_images, _,_ = reduce_image_dimensions(bob_images, target_dimension)

torch.save(reduced_images, 'data/reduced_images.pth')
torch.save(bob_labels, 'data/reduced_labels.pth')


Original image shape: (1000, 1, 28, 28)
Original feature space: 784
Target dimensions: 50
Flattened shape: (1000, 784)
After random projection: (1000, 50)
After scaling: (1000, 50)


In [9]:
#Perform clustering
from privade.clustering import kmeans_clustering

rep_set_size = 20

representative_set = kmeans_clustering(reduced_images, rep_set_size)

In [10]:
representative_points = reduced_images[representative_set]
rep_points = bob_images[representative_set]
rep_labels = bob_labels[representative_set]
torch.save(rep_points, 'data/rep_points.pth')
torch.save(rep_labels, 'data/rep_labels.pth')
dists = np.linalg.norm(reduced_images[:, None] - representative_points[None, :], axis=2)
min_dists = np.min(dists, axis=1)
max_min_distance = np.ceil(np.max(min_dists))
print("Maximum of the minimum distances:", max_min_distance)

Maximum of the minimum distances: 2.0


Next, we will prepare a setup for performing the challenge protocol with alice.

In [11]:
from privade.challenge_protocol import setup_challenge_protocol

# setup_challenge_protocol()

In [12]:
# from privade.challenge_protocol import create_proof, verify_proof

# M = 20 #challenge number

# #Alice randomly selects M points from the whole dataset
# indices = random.sample(range(N), M)
# print(indices)

# #For each data point do the Challenge Protocol

# for idx in indices:
#     #Find the index from representative_points which has the min distance from the selected point
#     selected_point = reduced_images[idx]
#     dists = np.linalg.norm(representative_points - selected_point, axis=1)
#     min_index = np.argmin(dists)
#     print(np.min(dists), min_index)
#     cp_data = {
#         "messageArray": selected_point.tolist(),
#         "idx": int(min_index),
#         "allPoints": representative_points.tolist(),
#         "d": int(max_min_distance),
#         "r": 0x12345678
#     }
#     assert len(selected_point.tolist()) == 50
#     assert len(representative_points.tolist()) == 20
    
#     proof_file = "proof.json"
    
#     assert create_proof(cp_data,proof_file)
    
#     assert verify_proof(proof_file)
    

## Part 2: Model Inference


The next part of the process involves model inference of A, B and C.

In [13]:
#Model A insecure inference
model_A = model_A.to(device)
model_A_output = model_A(torch.tensor(rep_points).to(device))


In [14]:
# Create a mapping from saved model structure to your simplified net structure
# def create_compatible_state_dict(saved_state_dict):
#     """
#     Convert the saved model_A state_dict to match the simple net structure
#     """
#     compatible_state = {}
    
#     # Mapping from saved keys to net keys
#     key_mapping = {
#         '0.0.weight': '0.weight',      # Conv2d weight
#         '0.0.bias': '0.bias',          # Conv2d bias  
#         '2.linear.weight': '3.weight', # Linear weight (after Flatten at index 2, Linear is at index 3)
#         '2.linear.bias': '3.bias'
#     }
    
#     for saved_key, net_key in key_mapping.items():
#         if saved_key in saved_state_dict:
#             compatible_state[net_key] = saved_state_dict[saved_key]
#             print(f"Mapped {saved_key} -> {net_key}: {saved_state_dict[saved_key].shape}")
    
#     return compatible_state

# # Load the saved weights and create compatible state dict
# saved_state = torch.load("/home/thomas/secure-data-valuation/notebooks/data/model_a.pth", weights_only=True)
# compatible_state = create_compatible_state_dict(saved_state)

# # Save the converted model for future use
# torch.save(compatible_state, "/home/thomas/secure-data-valuation/notebooks/data/model_a_compatible.pth")
# print("Saved compatible model to model_a_compatible.pth")

# class Square(nn.Module):
#     """Elementwise square activation: f(x) = x^2"""
#     def __init__(self, inplace: bool = False):
#         super().__init__()
#         self.inplace = inplace

#     def forward(self, x):
#         return x.mul_(x) if self.inplace else x * x

# net = nn.Sequential(
#     nn.Conv2d(1, 3, kernel_size=(5, 5), stride=(1, 1)) ,
#     Square(),
#     nn.Flatten(),
#     nn.Linear(1728,1728)
# )

# net.load_state_dict(torch.load("/home/thomas/secure-data-valuation/notebooks/data/model_a_compatible.pth"), strict=False)


In [15]:
#MPC 
from privade.mpc_inference import setup, inference

#Update the file

#Setup
# setup("model_a_inf")

# #Perform inference
# inference("model_a_inf")

In [16]:
#MOdel B: Bob inference
from privade.cnczk import choose_random_layers, collect_sequential_activations, get_layer, setup_zkp
import asyncio
#Plaintext inference
model_B.to(device)
model_B_output = model_B(model_A_output)

#Randomly choose layers:
layers = choose_random_layers(model_B,1)
print(layers)

for layer in layers:
    await setup_zkp(model_B, model_A_output, layer, 'pw')


[2]


In [17]:
import random
# Alice randomly chooses s number of points
s = 5
points_to_check = random.sample(range(len(model_A_output)), s)
selected_A_output = model_A_output[points_to_check]
selected_A_output.shape

torch.Size([5, 1728])

In [18]:
from privade.cnczk import prove_zkp, verify_zkp
for layer in layers:
    await prove_zkp(model_B,selected_A_output,layer)
    
for layer in layers:
    await verify_zkp(layer)

In [19]:
#Model C: Alice inference
#This is similar, just with different visibility settings for model weights and data.
model_C.to(device)
model_C_output = model_C(model_B_output)

In [20]:
#Randomly choose layers:
layers = choose_random_layers(model_C,1)
print(layers)

for layer in layers:
    await setup_zkp(model_C, model_B_output, layer, 'pi')

[2]


In [21]:
s = 5
points_to_check = random.sample(range(len(model_B_output)), s)
selected_B_output = model_B_output[points_to_check]
selected_B_output.shape

torch.Size([5, 900])

In [22]:
for layer in layers:
    await prove_zkp(model_C,selected_B_output,layer)
    
for layer in layers:
    await verify_zkp(layer)



## Part 3: Secure Scoring

In [23]:
#Prepare Alice and Bob's private input for MPC
#We use unreduced data for inference but reduced data for the diversity calculation here
points_to_submit = reduced_images[representative_set]
labels_to_submit = rep_labels

Bob_input = (points_to_submit, labels_to_submit)
Alice_input = model_C_output.cpu().detach()


from privade.scoring import prepare_inputs

assert prepare_inputs(Bob_input, Alice_input)

1000
(20, 10)
200
200


In [24]:
from privade.scoring import compile_program, run_mpc
assert compile_program()
assert run_mpc()

Default bit length for compilation: 63
Default security parameter for compilation: 40
Compiling file Programs/Source/multi_point_val.mpc


Writing to Programs/Bytecode/multi_point_val-FPDiv(2)_31_16-1.bc
Writing to Programs/Bytecode/multi_point_val-TruncPr(20)_47_16-3.bc
Writing to Programs/Bytecode/multi_point_val-FPDiv(1)_31_16-5.bc
Writing to Programs/Bytecode/multi_point_val-TruncPr(9)_47_16-6.bc
Writing to Programs/Bytecode/multi_point_val-TruncPr(5)_47_16-7.bc
Writing to Programs/Bytecode/multi_point_val-sqrt(17)_31_16-8.bc
Writing to Programs/Bytecode/multi_point_val-sqrt(16)_31_16-10.bc
Writing to Programs/Bytecode/multi_point_val-log2_fx(6)_31_16-11.bc
Writing to Programs/Bytecode/multi_point_val-TruncPr(6)_47_16-13.bc
Writing to Programs/Bytecode/multi_point_val-FPDiv(6)_31_16-14.bc
Writing to Programs/Bytecode/multi_point_val-log2_fx(2)_31_16-15.bc
Writing to Programs/Bytecode/multi_point_val-TruncPr(2)_47_16-16.bc
Compiled 100000 lines at Mon Aug 25 22:12:49 2025
Writing to Programs/Bytecode/multi_point_val-log2_fx(10)_31_16-17.bc
Writing to Programs/Bytecode/multi_point_val-FPDiv(10)_31_16-18.bc
Writing to Pr

Running /home/thomas/secure-data-valuation/MP-SPDZ/Scripts/../spdz2k-party.x 0 multi_point_val -pn 10510 -h localhost -N 2
Running /home/thomas/secure-data-valuation/MP-SPDZ/Scripts/../spdz2k-party.x 1 multi_point_val -pn 10510 -h localhost -N 2


Using SPDZ2k security parameter 64
Using statistical security parameter 40
Trying to run 64-bit computation
Diversity score: 0.141022
Uncertainty score: -9.2446
Loss score: -2.07898
Final Valuation: -3.56267
The following benchmarks are including preprocessing (offline phase).
Time = 91.789 seconds 
Data sent = 16912.6 MB in ~58299 rounds (party 0 only; use '-v' for more details)
Global data sent = 33825.2 MB (all parties)
This program might benefit from some protocol options.
Consider adding the following at the beginning of your code:
	program.use_edabit(True)


In [32]:
net = nn.Sequential(
    nn.Conv2d(3,6, kernel_size=5,padding=2) ,
    nn.ReLU(),
    nn.Flatten(),
    nn.Linear(6*32*32,1)
)

In [33]:
test_data = torch.load("/home/thomas/secure-data-valuation/experiments/data/rep_points.pth")
test_data.shape

/tmp/ipykernel_2691242/848170425.py:1: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  test_data = torch.load("/home/thomas/secure-data-valuation/experiments/data/rep_points.p

(50, 3, 32, 32)

In [34]:
test_data = torch.tensor(test_data)
net.cpu()
net(test_data)

tensor([[ 0.6380],
        [ 0.6291],
        [ 0.0153],
        [ 0.4325],
        [ 0.0856],
        [ 0.2585],
        [ 0.4718],
        [-0.3432],
        [-0.2133],
        [-0.0518],
        [ 0.2186],
        [ 0.0776],
        [ 0.0825],
        [ 0.0099],
        [-0.0864],
        [ 0.2313],
        [ 0.4419],
        [ 0.1991],
        [ 0.0649],
        [-0.1856],
        [ 0.3457],
        [ 0.2306],
        [ 0.1389],
        [ 0.3393],
        [ 0.2954],
        [-0.0723],
        [ 0.4399],
        [ 0.0780],
        [ 0.3022],
        [ 0.0304],
        [ 0.1894],
        [ 0.3558],
        [ 0.2477],
        [ 0.4580],
        [ 0.1345],
        [ 0.2933],
        [ 0.1693],
        [-0.0726],
        [ 0.6036],
        [ 0.0783],
        [ 0.1166],
        [ 0.3433],
        [-0.3444],
        [-0.4422],
        [ 0.0467],
        [ 0.2870],
        [ 0.3082],
        [ 0.1026],
        [-0.0658],
        [ 0.2478]], grad_fn=<AddmmBackward0>)